ДЗ №2
Задача
На занятии мы разобрали, как писать кернелы на Triton для простых поэлементных операций, таких как SiLU-Mul.

Ваша следующая задача — написать forward pass и backward pass (две разных функции) для операции Layer Normalization (torch.nn.LayerNorm).

Реализуйте forward pass.
Реализуйте backward pass.
Проверьте решение на корректность через torch.testing.assert_close.
Добавьте autotune.
Сравните скорость вашего решения со скоростью эталонного решения на PyTorch.
Эталонное решение на PyTorch:

In [ ]:
%pip -q install triton

In [ ]:
import torch

In [ ]:
def layernorm_forward_torch(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor, eps: float = 1e-5):
    mu = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, unbiased=False, keepdim=True)
    inv_std = torch.rsqrt(var + eps)
    normalized = (x - mu) * inv_std
    return normalized * weight + bias

Подсказка: Пусть дана матрица [M; N], где M - количество элементов, а N - hidden size; в гриде вы будете запускаться по оси M, имея доступ ко всем N в рамках выбранных (или выбранного) M, - это знание упростит подсчёт статистики.

Подсказка: Для аккумуляции градиентов в backward pass вам может понадобиться функция tl.atomic_add: документация.

Сдача
Файл с исходным кодом в виде ссылки на путь в GitHub-репозитории нужно отправить мне в личку в телеграм.

Если вы сделали бенчмарк, приложите к репозиторию/сообщению скриншот графика или просто цифры из текстового репорта пропускной способности.

Баллы
В зависимости от того, какую часть задания вы сделаете:

3 балла - только forward pass, с проверкой корректности;
4 балла - forward pass и backward pass, с проверкой корректности;
5 баллов - forward pass и backward pass, с проверкой корректности, автотюном и бенчмарком.
Дедлайн
Сдавать и корректировать решение можно до 29 мая.

### Решение

Один Triton-блок обрабатывает строку `[N]`. Статистики считаются внутри блока, градиенты по `weight`/`bias` накапливаются через `atomic_add`.

In [ ]:
import torch
import triton
import triton.language as tl

assert torch.cuda.is_available(), 'нужна CUDA'
DEV = torch.device('cuda')
print('device:', DEV)

In [ ]:
LN_TUNING = [
    triton.Config({}, num_warps=2),
    triton.Config({}, num_warps=4),
    triton.Config({}, num_warps=8),
    triton.Config({}, num_warps=16),
]


@triton.autotune(configs=LN_TUNING, key=['n_cols'])
@triton.jit
def ln_fwd_kernel(
    inp_ptr, out_ptr, gamma_ptr, beta_ptr,
    n_rows, n_cols,
    stride_in_row, stride_in_col,
    stride_out_row, stride_out_col,
    eps,
    BLOCK: tl.constexpr,
):
    pid = tl.program_id(0)
    lanes = tl.arange(0, BLOCK)

    row_off_in = pid * stride_in_row
    row_off_out = pid * stride_out_row
    col_off = lanes * stride_in_col

    ptrs_in = inp_ptr + row_off_in + col_off
    ptrs_out = out_ptr + row_off_out + lanes * stride_out_col
    active = lanes < n_cols

    row = tl.load(ptrs_in, mask=active, other=0.0)
    n = tl.full((), n_cols, tl.float32)

    mu = tl.sum(row, axis=0) / n
    centered = row - mu
    inv_std = tl.rsqrt(tl.sum(centered * centered, axis=0) / n + eps)
    normed = centered * inv_std

    gamma = tl.load(gamma_ptr + lanes, mask=active, other=1.0)
    beta = tl.load(beta_ptr + lanes, mask=active, other=0.0)
    tl.store(ptrs_out, normed * gamma + beta, mask=active)


@triton.autotune(configs=LN_TUNING, key=['n_cols'])
@triton.jit
def ln_bwd_kernel(
    grad_out_ptr, inp_ptr, grad_inp_ptr, grad_gamma_ptr, grad_beta_ptr, gamma_ptr,
    n_rows, n_cols,
    stride_go_row, stride_go_col,
    stride_in_row, stride_in_col,
    stride_gi_row, stride_gi_col,
    eps,
    BLOCK: tl.constexpr,
):
    pid = tl.program_id(0)
    lanes = tl.arange(0, BLOCK)
    active = lanes < n_cols
    n = tl.full((), n_cols, tl.float32)

    off_in = pid * stride_in_row + lanes * stride_in_col
    off_go = pid * stride_go_row + lanes * stride_go_col
    off_gi = pid * stride_gi_row + lanes * stride_gi_col

    x = tl.load(inp_ptr + off_in, mask=active, other=0.0)
    go = tl.load(grad_out_ptr + off_go, mask=active, other=0.0)
    gamma = tl.load(gamma_ptr + lanes, mask=active, other=1.0)

    mu = tl.sum(x, axis=0) / n
    centered = x - mu
    inv_std = tl.rsqrt(tl.sum(centered * centered, axis=0) / n + eps)
    normed = centered * inv_std

    d_norm = go * gamma
    sum_d_norm = tl.sum(d_norm, axis=0)
    sum_d_norm_xhat = tl.sum(d_norm * normed, axis=0)
    scale = inv_std / n
    grad_in = scale * (n * d_norm - sum_d_norm - normed * sum_d_norm_xhat)

    tl.store(grad_inp_ptr + off_gi, grad_in, mask=active)
    tl.atomic_add(grad_gamma_ptr + lanes, go * normed, mask=active)
    tl.atomic_add(grad_beta_ptr + lanes, go, mask=active)

In [ ]:
BLOCK_LIMIT = 2048


def _block_for(hidden: int) -> int:
    bs = triton.next_power_of_2(hidden)
    if bs > BLOCK_LIMIT:
        raise ValueError(f'hidden={hidden} слишком большой для текущего BLOCK')
    return bs


def triton_layer_norm(x, gamma, beta, eps=1e-5, block=None):
    if not x.is_cuda:
        raise RuntimeError('только CUDA')
    rows, cols = x.shape
    block = block or _block_for(cols)
    out = torch.empty_like(x)
    x, out, gamma, beta = x.contiguous(), out.contiguous(), gamma.contiguous(), beta.contiguous()
    ln_fwd_kernel[(rows,)](
        x, out, gamma, beta,
        rows, cols,
        x.stride(0), x.stride(1),
        out.stride(0), out.stride(1),
        eps,
        BLOCK=block,
    )
    return out


def triton_layer_norm_bwd(grad_out, x, gamma, eps=1e-5, block=None):
    if not (grad_out.is_cuda and x.is_cuda):
        raise RuntimeError('только CUDA')
    rows, cols = x.shape
    block = block or _block_for(cols)
    grad_in = torch.empty_like(x)
    grad_gamma = torch.zeros_like(gamma)
    grad_beta = torch.zeros_like(gamma)
    x, grad_out = x.contiguous(), grad_out.contiguous()
    grad_in = grad_in.contiguous()
    gamma = gamma.contiguous()
    ln_bwd_kernel[(rows,)](
        grad_out, x, grad_in, grad_gamma, grad_beta, gamma,
        rows, cols,
        grad_out.stride(0), grad_out.stride(1),
        x.stride(0), x.stride(1),
        grad_in.stride(0), grad_in.stride(1),
        eps,
        BLOCK=block,
    )
    return grad_in, grad_gamma, grad_beta

In [ ]:
def check_against_pytorch():
    torch.manual_seed(42)
    rows, cols = 256, 512
    eps = 1e-5
    tol = dict(rtol=1e-2, atol=1e-2)

    x = torch.randn(rows, cols, device=DEV, requires_grad=True)
    gamma = torch.randn(cols, device=DEV, requires_grad=True)
    beta = torch.randn(cols, device=DEV, requires_grad=True)

    x_ref = x.detach().clone().requires_grad_()
    g_ref = gamma.detach().clone().requires_grad_()
    b_ref = beta.detach().clone().requires_grad_()
    y_ref = torch.nn.functional.layer_norm(x_ref, (cols,), g_ref, b_ref, eps=eps)

    y = triton_layer_norm(x, gamma, beta, eps=eps)
    grad_y = torch.randn_like(y_ref)

    dx_ref, dg_ref, db_ref = torch.autograd.grad(y_ref, (x_ref, g_ref, b_ref), grad_outputs=grad_y)
    dx, dg, db = triton_layer_norm_bwd(grad_y, x, gamma, eps=eps)

    torch.testing.assert_close(y, y_ref, **tol)
    torch.testing.assert_close(dx, dx_ref, **tol)
    torch.testing.assert_close(dg, dg_ref, **tol)
    torch.testing.assert_close(db, db_ref, **tol)
    print('forward/backward совпали с F.layer_norm')


check_against_pytorch()

In [ ]:
def _timed_ms(fn, warmup=10, repeat=50):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    start, end = torch.cuda.Event(True), torch.cuda.Event(True)
    start.record()
    for _ in range(repeat):
        fn()
    end.record()
    torch.cuda.synchronize()
    return start.elapsed_time(end) / repeat


def compare_ln_speed(rows, cols, warmup=10, repeat=50):
    x = torch.randn(rows, cols, device=DEV)
    gamma = torch.randn(cols, device=DEV)
    beta = torch.randn(cols, device=DEV)
    elems = rows * cols

    t_custom = _timed_ms(lambda: triton_layer_norm(x, gamma, beta), warmup, repeat)
    t_native = _timed_ms(
        lambda: torch.nn.functional.layer_norm(x, (cols,), gamma, beta),
        warmup,
        repeat,
    )

    def gbps(ms):
        return elems / (ms * 1e-3) / 1e9

    print(f'shape=({rows}, {cols})')
    print(f'triton:  {t_custom:.3f} ms  ({gbps(t_custom):.2f} Gelem/s)')
    print(f'pytorch: {t_native:.3f} ms  ({gbps(t_native):.2f} Gelem/s)')


compare_ln_speed(1024, 1024)